In [7]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime
import ast
headers = {"User-Agent": "MyDataResearchProject-dev-1.0.0", "Accept": "application/ld+json"}

In [2]:
df = pd.read_csv("second_readings_found.csv")

In [4]:
df.head(20)

,meeting_id,vot_itm_id,title
0,MTG-PL-2014-04-03,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64,<structuredLabel><label>Recommandation pour la...
1,MTG-PL-2014-04-03,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1,<structuredLabel><title>Régime communautaire d...
2,MTG-PL-2016-04-14,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522221-1,<structuredLabel><title>Protection des personn...
3,MTG-PL-2016-04-14,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522222-2,<structuredLabel><title>Protection des personn...
4,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520211-8,<structuredLabel><title>Interopérabilité du sy...
5,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520212-9,<structuredLabel><title>Sécurité ferroviaire</...
6,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520210-7,<structuredLabel><title>Agence de l’Union euro...
7,MTG-PL-2016-06-09,eli/dl/event/MTG-PL-2016-06-09-VOT-ITM-540102-4,<structuredLabel><title>Favoriser la libre cir...
8,MTG-PL-2025-10-23,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,<structuredLabel><title>Soil Monitoring and Re...
9,MTG-PL-2025-10-23,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337,<structuredLabel><title>Preventing plastic pel...


In [ ]:
### OBS MADE OBSOLETE BY SCRIPT BELOW###

#decisions_json_folder = "decisions_json_dumps"
os.makedirs(decisions_json_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

all_extracted_decisions = []

# Assuming df_second_readings is the dataframe from our previous script
for vote_id in df["vot_itm_id"]:
    print(f"Fetching specific vote: {vote_id}...")
    
    # 1. Hit the specific VOT-ITM directly
    url_1 = f"https://data.europarl.europa.eu/{vote_id}"
    
    try:
        # Remember our timeout safety net!
        response_1 = requests.get(url_1, headers=headers)
        
        if response_1.status_code == 200:
            vote_data = response_1.json()
            
            # The direct ELI resolution usually returns the item inside a "data" array
            if "data" in vote_data and len(vote_data["data"]) > 0:
                vote_item = vote_data["data"][0]
                
                # 2. Look for the DEC codes inside "consists_of"
                if "consists_of" in vote_item and isinstance(vote_item["consists_of"], list):
                    
                    for decision_id in vote_item["consists_of"]:
                        if "-DEC-" not in decision_id:
                            continue
                        
                        print(f"  -> Found decision: {decision_id}. Fetching metadata...")
                        
                        # 3. Hit the specific DEC item directly
                        url_2 = f"https://data.europarl.europa.eu/{decision_id}"
                        response_2 = requests.get(url_2, headers=headers, timeout=15)
                        
                        if response_2.status_code == 200:
                            decision_data = response_2.json()
                            
                            # Save the DEC json dump
                            safe_filename = decision_id.split("/")[-1] + ".json"
                            file_path = os.path.join(decisions_json_folder, safe_filename)
                            
                            with open(file_path, "w", encoding="utf-8") as f:
                                json.dump(decision_data, f, indent=4)
                            
                            # 4. Extract the juicy metadata
                            if "data" in decision_data and len(decision_data["data"]) > 0:
                                target_dict = decision_data["data"][0]
                                
                                record = {
                                    "vote_id": vote_id,
                                    "decision_id": decision_id,
                                    "start_date": target_dict.get("activity_start_date"),
                                    "method": target_dict.get("decision_method"),
                                    "outcome": target_dict.get("decision_outcome"),
                                    "attendees": target_dict.get("number_of_attendees"),
                                    "votes_favor": target_dict.get("number_of_votes_favor"),
                                    "votes_against": target_dict.get("number_of_votes_against")
                                }
                                all_extracted_decisions.append(record)
                        else:
                            print(f"  -> Failed to fetch DEC {decision_id}: {response_2.status_code}")
                            
                        time.sleep(0.5) # Pause between DEC requests
            else:
                print(f"No valid 'data' found for {vote_id}")
        else:
            print(f"Failed to fetch {vote_id}: Status {response_1.status_code}")
            
    except requests.exceptions.Timeout:
        print(f"Warning: Request timed out for {vote_id}. Skipping...")
    except Exception as e:
        print(f"Error on {vote_id}: {e}")
        
    time.sleep(0.5) # Pause between VOT-ITM requests

# 5. Build the final clean dataframe
df_decisions = pd.DataFrame(all_extracted_decisions)
print("\n--- EXTRACTION COMPLETE ---")
if not df_decisions.empty:
    print(df_decisions.head())

Fetching specific vote: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64...
Fetching specific vote: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522221-1...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522222-2...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520211-8...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520212-9...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520210-7...
Fetching specific vote: eli/dl/event/MTG-PL-2016-06-09-VOT-ITM-540102-4...
Fetching specific vote: eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331...
  -> Found decision: eli/dl/event/MTG-PL-2025-10-23-DEC-180520. Fetching metadata...
Fetching specific vote: eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337...
  -> Found decision: eli/dl/event/MTG-PL-2025-10-23-DEC-180605. Fetching metadata...
Fetching specific vote: eli/dl/event/MTG-PL-2025-11-13-VOT-ITM-975467...
  -> Found

In [6]:
#Since we are getting many DEC files for every vote item, we will now comb through them to only catch the ones with "rule 68" or 69
#These are the actual absolute majority votes

decisions_json_folder = "decisions_json_dumps"
votes_json_folder = "votes_json_dumps" # <-- NEW FOLDER FOR THE FIRST ROUND
os.makedirs(decisions_json_folder, exist_ok=True)
os.makedirs(votes_json_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

all_extracted_decisions = []

# Assuming df_second_readings is the dataframe from our previous script
for vote_id in df["vot_itm_id"]:
    print(f"Fetching specific vote: {vote_id}...")
    
    # 1. Hit the specific VOT-ITM directly
    url_1 = f"https://data.europarl.europa.eu/{vote_id}"
    
    try:
        response_1 = requests.get(url_1, headers=headers, timeout=15)
        
        if response_1.status_code == 200:
            vote_data = response_1.json()
            
            # --- NEW SAVE BLOCK: Save the first round (VOT-ITM) JSON ---
            safe_vote_filename = vote_id.replace("/", "_") + ".json"
            vote_file_path = os.path.join(votes_json_folder, safe_vote_filename)
            try:
                with open(vote_file_path, "w", encoding="utf-8") as f:
                    json.dump(vote_data, f, indent=4)
            except Exception as save_err:
                print(f"  -> Warning: Could not save vote file {safe_vote_filename}: {save_err}")
            # -----------------------------------------------------------
            
            # The direct ELI resolution usually returns the item inside a "data" array
            if "data" in vote_data and len(vote_data["data"]) > 0:
                vote_item = vote_data["data"][0]
                
                # 2. Look for the DEC codes inside "consists_of"
                if "consists_of" in vote_item and isinstance(vote_item["consists_of"], list):
                    
                    for decision_id in vote_item["consists_of"]:
                        if "-DEC-" not in decision_id:
                            continue
                        
                        print(f"  -> Found decision: {decision_id}. Fetching metadata...")
                        
                        # 3. Hit the specific DEC item directly
                        url_2 = f"https://data.europarl.europa.eu/{decision_id}"
                        response_2 = requests.get(url_2, headers=headers, timeout=15)
                        
                        if response_2.status_code == 200:
                            decision_data = response_2.json()
                            
                            # Save the DEC json dump
                            safe_dec_filename = decision_id.split("/")[-1] + ".json"
                            file_path = os.path.join(decisions_json_folder, safe_dec_filename)
                            
                            with open(file_path, "w", encoding="utf-8") as f:
                                json.dump(decision_data, f, indent=4)
                            
                            # 4. Extract the juicy metadata
                            if "data" in decision_data and len(decision_data["data"]) > 0:
                                target_dict = decision_data["data"][0]
                                
                                heading_en = target_dict.get("headingLabel", {}).get("en", "")

                                # 2. Flag if it is a Rule 68 (Rejection) or Rule 69 (Amendment) vote
                                is_absolute_majority = "Rule 68" in heading_en or "Rule 69" in heading_en

                                # 3. Add it to your record dictionary
                                record = {
                                    "vote_id": vote_id,
                                    "decision_id": decision_id,
                                    "start_date": target_dict.get("activity_start_date"),
                                    "method": target_dict.get("decision_method"),
                                    "outcome": target_dict.get("decision_outcome"),
                                    "attendees": target_dict.get("number_of_attendees"),
                                    "votes_favor": target_dict.get("number_of_votes_favor"),
                                    "votes_against": target_dict.get("number_of_votes_against"),
                                    "heading": heading_en,                         # <-- Save the label for context
                                    "absolute_majority": is_absolute_majority      # <-- Your new boolean flag!
                                }
                                all_extracted_decisions.append(record)
                        else:
                            print(f"  -> Failed to fetch DEC {decision_id}: {response_2.status_code}")
                            
                        time.sleep(0.5) # Pause between DEC requests
            else:
                print(f"No valid 'data' found for {vote_id}")
        else:
            print(f"Failed to fetch {vote_id}: Status {response_1.status_code}")
            
    except requests.exceptions.Timeout:
        print(f"Warning: Request timed out for {vote_id}. Skipping...")
    except Exception as e:
        print(f"Error on {vote_id}: {e}")
        
    time.sleep(0.5) # Pause between VOT-ITM requests

# 5. Build the final clean dataframe
df_decisions = pd.DataFrame(all_extracted_decisions)
print("\n--- EXTRACTION COMPLETE ---")
if not df_decisions.empty:
    print(df_decisions.head())

Fetching specific vote: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64...
Fetching specific vote: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522221-1...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522222-2...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520211-8...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520212-9...
Fetching specific vote: eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520210-7...
Fetching specific vote: eli/dl/event/MTG-PL-2016-06-09-VOT-ITM-540102-4...
Fetching specific vote: eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331...
  -> Found decision: eli/dl/event/MTG-PL-2025-10-23-DEC-180520. Fetching metadata...
Fetching specific vote: eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337...
  -> Found decision: eli/dl/event/MTG-PL-2025-10-23-DEC-180605. Fetching metadata...
Fetching specific vote: eli/dl/event/MTG-PL-2025-11-13-VOT-ITM-975467...
  -> Found

In [9]:
print(len(df_decisions))
df_decisions.head(10)

37


,vote_id,decision_id,start_date,method,outcome,attendees,votes_favor,votes_against,heading,absolute_majority
0,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,eli/dl/event/MTG-PL-2025-10-23-DEC-180520,2025-10-23T13:08:06+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,571.0,220.0,341.0,Proposal to reject the Council position,False
1,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337,eli/dl/event/MTG-PL-2025-10-23-DEC-180605,NaN,NaN,NaN,NaN,NaN,NaN,,False
2,eli/dl/event/MTG-PL-2025-11-13-VOT-ITM-975467,eli/dl/event/MTG-PL-2025-11-13-DEC-181469,NaN,NaN,NaN,NaN,NaN,NaN,,False
3,eli/dl/event/MTG-PL-2025-11-13-VOT-ITM-975415,eli/dl/event/MTG-PL-2025-11-13-DEC-181470,NaN,NaN,NaN,NaN,NaN,NaN,,False
4,eli/dl/event/MTG-PL-2026-01-22-VOT-ITM-982162,eli/dl/event/MTG-PL-2026-01-22-DEC-184076,NaN,NaN,NaN,NaN,NaN,NaN,,False
5,eli/dl/event/MTG-PL-2026-03-26-VOT-ITM-987077,eli/dl/event/MTG-PL-2026-03-26-DEC-189897,NaN,NaN,NaN,NaN,NaN,NaN,,False
6,eli/dl/event/MTG-PL-2026-03-26-VOT-ITM-987073,eli/dl/event/MTG-PL-2026-03-26-DEC-189900,NaN,NaN,NaN,NaN,NaN,NaN,,False
7,eli/dl/event/MTG-PL-2026-03-26-VOT-ITM-987078,eli/dl/event/MTG-PL-2026-03-26-DEC-189899,NaN,NaN,NaN,NaN,NaN,NaN,,False
8,eli/dl/event/MTG-PL-2026-03-26-VOT-ITM-987079,eli/dl/event/MTG-PL-2026-03-26-DEC-189898,NaN,NaN,NaN,NaN,NaN,NaN,,False
9,eli/dl/event/MTG-PL-2026-07-09-VOT-ITM-993015,eli/dl/event/MTG-PL-2026-07-09-DEC-195789,2026-07-09T13:26:27+02:00,def/ep-decision-methods/VOTE_ELECTRONIC_ROLLCALL,def/ep-statuses/REJECTED,605.0,175.0,408.0,Draft legislative act,False


In [ ]:
###OBS THIS ONE IS OBSOLETE ##

#Okay now we want to find the vote results from 2014 and 2016, which are formatted differently.
#We have to go through layers of jsons and then ultimately to a docx file, where the outcome will be recorded
import re
import docx
# 1. Setup Folders
#older_votes_folder = "votes_json_dumps/older_votes"
docs_folder = "downloaded_docs"
os.makedirs(docs_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en"
}

extracted_older_data = []

# --- HELPER FUNCTIONS ---

def get_docx_text(file_path):
    """Extracts text from a DOCX file, reading both paragraphs and tables in order."""
    doc = docx.Document(file_path)
    text_blocks = []
    
    # We have to read the raw XML blocks so we don't skip the tables!
    for block in doc.element.body:
        if block.tag.endswith('p'): # It's a paragraph
            text_blocks.append(block.text or "")
        elif block.tag.endswith('tbl'): # It's a table
            for row in block.xpath('.//w:tr'):
                row_text = []
                for cell in row.xpath('.//w:tc'):
                    cell_text = "".join([node.text for node in cell.xpath('.//w:t') if node.text])
                    row_text.append(cell_text)
                text_blocks.append(" | ".join(row_text))
                
    return "\n".join(text_blocks)

def parse_older_vote(full_text, item_number):
    """Slices the document for the specific item and searches for the vote outcome."""
    outcome_data = {
        "method": "Unknown",
        "outcome": "Unknown",
        "votes_favor": None,
        "votes_against": None,
        "abstentions": None
    }
    
    # Slice the text: Grab everything from "7. " to "8. " (or the end of the document)
    pattern = rf"(?m)^{item_number}\.\s.*?(?=^{int(item_number)+1}\.\s|\Z)"
    match = re.search(pattern, full_text, re.DOTALL)
    
    if not match:
        return outcome_data
        
    text_chunk = match.group(0)
    
    # Check for auto-approval (Second Reading without amendment)
    if "Approval without vote" in text_chunk:
        outcome_data["method"] = "Auto-adopted"
        outcome_data["outcome"] = "Approved"
        return outcome_data
        
    # Look for the adopted/rejected signs
    if " | + | " in text_chunk:
        outcome_data["outcome"] = "Adopted"
    elif " | - | " in text_chunk:
        outcome_data["outcome"] = "Rejected"
        
    # Use Regex to find the vote numbers (e.g., "487, 38, 2")
    vote_match = re.search(r'(\d+),\s*(\d+),\s*(\d+)', text_chunk)
    if vote_match:
        outcome_data["method"] = "Roll Call / Electronic"
        outcome_data["votes_favor"] = int(vote_match.group(1))
        outcome_data["votes_against"] = int(vote_match.group(2))
        outcome_data["abstentions"] = int(vote_match.group(3))
        
    return outcome_data

# --- MAIN LOOP ---

print("Scanning older JSON files...")

for filename in os.listdir(older_votes_folder):
    if not filename.endswith(".json"):
        continue
        
    filepath = os.path.join(older_votes_folder, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        vote_data = json.load(f)
        
    # Make sure valid data exists
    if "data" not in vote_data or len(vote_data["data"]) == 0:
        continue
        
    item_data = vote_data["data"][0]
    vote_id = item_data.get("id")
    
    # Find the link to the Procès-Verbal (PV) minutes document
    pv_links = item_data.get("recorded_in_a_realization_of", [])
    if not pv_links:
        print(f"No PV minutes linked for {vote_id}")
        continue
        
    pv_item_link = pv_links[0] # e.g., "eli/dl/doc/PV-8-2016-04-28-VOT-ITM-007"
    
    # Derive the Master Document link and the item number
    master_doc_id = pv_item_link.split("-ITM")[0]
    raw_item_num = pv_item_link.split("-ITM-")[1]
    item_number = str(int(raw_item_num)) # Turns "007" into "7"
    
    print(f"\nProcessing Item {item_number} from {master_doc_id}...")
    
    # Fetch the master document JSON to find the DOCX download link
    doc_url = f"https://data.europarl.europa.eu/{master_doc_id}"
    try:
        doc_resp = requests.get(doc_url, headers=headers, timeout=15)
        if doc_resp.status_code != 200:
            print(f"  -> Failed to fetch master doc info: {doc_resp.status_code}")
            continue
            
        master_doc_data = doc_resp.json()
        docx_download_path = None
        
        # Dig through the JSON to find the English DOCX link
        for item in master_doc_data.get("data", []):
            for realization in item.get("is_realized_by", []):
                if "language/ENG" in realization.get("language", ""):
                    for embodiment in realization.get("is_embodied_by", []):
                        if "file-type/DOCX" in embodiment.get("format", ""):
                            docx_download_path = embodiment.get("is_exemplified_by")
                            break
                            
        if not docx_download_path:
            print("  -> Could not find an English DOCX link.")
            continue
            
        # Download the DOCX file
        docx_url = f"https://data.europarl.europa.eu/{docx_download_path}"
        safe_doc_name = docx_download_path.split("/")[-1]
        local_doc_path = os.path.join(docs_folder, safe_doc_name)
        
        # Only download if we haven't already saved it for a previous vote!
        if not os.path.exists(local_doc_path):
            print(f"  -> Downloading {safe_doc_name}...")
            file_resp = requests.get(docx_url, timeout=30)
            with open(local_doc_path, "wb") as doc_file:
                doc_file.write(file_resp.content)
        else:
            print(f"  -> {safe_doc_name} already downloaded. Skipping download.")
            
        # Parse the document
        full_document_text = get_docx_text(local_doc_path)
        result = parse_older_vote(full_document_text, item_number)
        
        print(f"  -> Outcome: {result['outcome']} | Method: {result['method']}")
        if result['votes_favor'] is not None:
             print(f"  -> Votes: {result['votes_favor']} (+), {result['votes_against']} (-), {result['abstentions']} (O)")
        
        # Save to our final list
        extracted_older_data.append({
            "vote_id": vote_id,
            "master_doc": master_doc_id,
            "item_number": item_number,
            "method": result["method"],
            "outcome": result["outcome"],
            "votes_favor": result["votes_favor"],
            "votes_against": result["votes_against"],
            "abstentions": result["abstentions"]
        })
        
    except Exception as e:
        print(f"  -> Error processing {vote_id}: {e}")
        
    time.sleep(1) # Respect the API!

df_older_votes = pd.DataFrame(extracted_older_data)
print("\n--- ALL OLDER VOTES EXTRACTED ---")
if not df_older_votes.empty:
    print(df_older_votes.head())

Scanning older JSON files...

Processing Item 64 from eli/dl/doc/PV-7-2014-04-03-VOT...
  -> Could not find an English DOCX link.

Processing Item 1 from eli/dl/doc/PV-7-2014-04-03-VOT...
  -> Could not find an English DOCX link.

Processing Item 1 from eli/dl/doc/PV-8-2016-04-14-VOT...
  -> Downloading PV-8-2016-04-14-VOT-FNL_en.docx...
  -> Outcome: Unknown | Method: Unknown

Processing Item 2 from eli/dl/doc/PV-8-2016-04-14-VOT...
  -> PV-8-2016-04-14-VOT-FNL_en.docx already downloaded. Skipping download.
  -> Outcome: Unknown | Method: Unknown

Processing Item 7 from eli/dl/doc/PV-8-2016-04-28-VOT...
  -> Downloading PV-8-2016-04-28-VOT-FNL_en.docx...
  -> Outcome: Unknown | Method: Unknown

Processing Item 8 from eli/dl/doc/PV-8-2016-04-28-VOT...
  -> PV-8-2016-04-28-VOT-FNL_en.docx already downloaded. Skipping download.
  -> Outcome: Unknown | Method: Unknown

Processing Item 9 from eli/dl/doc/PV-8-2016-04-28-VOT...
  -> PV-8-2016-04-28-VOT-FNL_en.docx already downloaded. Skippin

In [ ]:
#We debug why it didn't catch any of the 2014 or 2016 votes

def get_docx_text(file_path):
    doc = docx.Document(file_path)
    text_blocks = []
    for block in doc.element.body:
        if block.tag.endswith('p'): 
            text_blocks.append(block.text or "")
        elif block.tag.endswith('tbl'): 
            for row in block.xpath('.//w:tr'):
                row_text = []
                for cell in row.xpath('.//w:tc'):
                    cell_text = "".join([node.text for node in cell.xpath('.//w:t') if node.text])
                    row_text.append(cell_text)
                text_blocks.append(" | ".join(row_text))
    return "\n".join(text_blocks)

# 1. Print the raw 2016 text
try:
    print("=== RAW 2016 DOCX TEXT ===")
    text = get_docx_text("downloaded_docs/PV-8-2016-04-28-VOT-FNL_en.docx")
    idx = text.find("EU Agency for Railways")
    # Print a chunk of text around the target item
    print(text[max(0, idx-100):idx+500])
except Exception as e:
    print("Could not read DOCX:", e)

# 2. Check 2014 file formats
print("\n=== 2014 FILE FORMATS ===")
headers = {"Accept": "application/ld+json", "Accept-Language": "en"}
resp = requests.get("https://data.europarl.europa.eu/eli/dl/doc/PV-7-2014-04-03-VOT", headers=headers)
if resp.status_code == 200:
    for item in resp.json().get("data", []):
        for real in item.get("is_realized_by", []):
            if "language/ENG" in real.get("language", ""):
                for emb in real.get("is_embodied_by", []):
                    print(emb.get("format"))

=== RAW 2016 DOCX TEXT ===
tation by the EMPL Committee

Subject | RCV etc. | Vote | RCV/EV – remarks
single vote |  | + | 



EU Agency for Railways ***II
Recommendation for second reading: Roberts Zīle (A8-0073/2016) (qualified majority to reject the Council position)
Subject | Am No | Author | RCV etc. | Vote | RCV/EV – remarks
Proposal to reject the Council proposal | 1 | EFDD |  | - | 
Council position | Approval without vote

Miscellaneous
The Council had informed Parliament that it had made a technical correction (change of a date) in Article 65(10) of its position in order to bring the text into line with the pr

=== 2014 FILE FORMATS ===
http://publications.europa.eu/resource/authority/file-type/XML
http://publications.europa.eu/resource/authority/file-type/PDF


In [21]:
older_votes_folder = "votes_json_dumps/older_votes"
docs_folder = "downloaded_docs"
os.makedirs(docs_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en"
}

extracted_older_data = []

# --- NEW HELPER FUNCTION ---

def parse_older_vote_xml(xml_data, item_number):
    """Isolates the specific XML block for the item, extracts tags natively, and finds the outcome."""
    outcome_data = {
        "method": "Unknown",
        "outcome": "Unknown",
        "votes_favor": None,
        "votes_against": None,
        "abstentions": None
    }
    
    # 1. Grab everything inside the specific <Vote.Result Number="X."> tag
    pattern = rf'<Vote\.Result Number="{item_number}\.?">(.*?)</Vote\.Result>'
    match = re.search(pattern, xml_data, re.DOTALL | re.IGNORECASE)
    
    if not match:
        # If the item literally doesn't exist in the XML (like Item 64)
        return outcome_data
        
    raw_block = match.group(1)
    
    # 2. Extract Numbers BEFORE stripping tags (using the dedicated XML tags)
    # We use findall and grab the last one [-1] to ensure we get the final vote of the item
    for_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.For>(\d+)<', raw_block)
    against_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.Against>(\d+)<', raw_block)
    abs_matches = re.findall(r'<Vote\.Result\.Table\.TotalVote\.Abstention>(\d+)<', raw_block)
    
    if for_matches and against_matches and abs_matches:
        outcome_data["method"] = "Roll Call / Electronic"
        outcome_data["votes_favor"] = int(for_matches[-1])
        outcome_data["votes_against"] = int(against_matches[-1])
        outcome_data["abstentions"] = int(abs_matches[-1])
        
    # 3. Clean the XML tags out to evaluate the text outcome
    text_chunk = re.sub(r'<[^>]+>', ' ', raw_block)
    text_chunk = re.sub(r'\s+', ' ', text_chunk).strip()
    
    # 4. Determine Outcome
    if "Declared approved" in text_chunk or "Approval without vote" in text_chunk or "without vote" in text_chunk.lower():
        outcome_data["outcome"] = "Approved"
        # Only set to Auto-adopted if there was no roll-call vote on an amendment/rejection
        if outcome_data["method"] == "Unknown":
            outcome_data["method"] = "Auto-adopted"
    else:
        # Determine outcome based on isolated "+" or "-" symbols
        pluses = [m.start() for m in re.finditer(r'(?<=\s)\+(?=\s)', text_chunk)]
        minuses = [m.start() for m in re.finditer(r'(?<=\s)-(?=\s)', text_chunk)]
        
        last_plus = pluses[-1] if pluses else -1
        last_minus = minuses[-1] if minuses else -1
        
        if last_plus > last_minus:
            outcome_data["outcome"] = "Adopted"
        elif last_minus > last_plus:
            outcome_data["outcome"] = "Rejected"
        
    return outcome_data

# --- MAIN LOOP ---

print("Scanning older JSON files...")

for filename in os.listdir(older_votes_folder):
    if not filename.endswith(".json"):
        continue
        
    filepath = os.path.join(older_votes_folder, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        vote_data = json.load(f)
        
    # Make sure valid data exists
    if "data" not in vote_data or len(vote_data["data"]) == 0:
        continue
        
    item_data = vote_data["data"][0]
    vote_id = item_data.get("id")
    
    # Find the link to the Procès-Verbal (PV) minutes document
    pv_links = item_data.get("recorded_in_a_realization_of", [])
    if not pv_links:
        print(f"No PV minutes linked for {vote_id}")
        continue
        
    pv_item_link = pv_links[0]
    
    # Derive the Master Document link and the item number
    master_doc_id = pv_item_link.split("-ITM")[0]
    raw_item_num = pv_item_link.split("-ITM-")[1]
    item_number = str(int(raw_item_num)) 
    
    print(f"\nProcessing Item {item_number} from {master_doc_id}...")
    
    doc_url = f"https://data.europarl.europa.eu/{master_doc_id}"
    try:
        doc_resp = requests.get(doc_url, headers=headers, timeout=15)
        if doc_resp.status_code != 200:
            print(f"  -> Failed to fetch master doc info: {doc_resp.status_code}")
            continue
            
        master_doc_data = doc_resp.json()
        xml_download_path = None
        
        # Dig through the JSON to find the English XML link
        for item in master_doc_data.get("data", []):
            for realization in item.get("is_realized_by", []):
                if "language/ENG" in realization.get("language", ""):
                    for embodiment in realization.get("is_embodied_by", []):
                        if "file-type/XML" in embodiment.get("format", ""):
                            xml_download_path = embodiment.get("is_exemplified_by")
                            break
                            
        if not xml_download_path:
            print("  -> Could not find an English XML link.")
            continue
            
        # Download the XML file
        xml_url = f"https://data.europarl.europa.eu/{xml_download_path}"
        safe_doc_name = xml_download_path.split("/")[-1]
        local_doc_path = os.path.join(docs_folder, safe_doc_name)
        
        if not os.path.exists(local_doc_path):
            print(f"  -> Downloading {safe_doc_name}...")
            file_resp = requests.get(xml_url, timeout=30)
            with open(local_doc_path, "wb") as doc_file:
                doc_file.write(file_resp.content)
        else:
            print(f"  -> {safe_doc_name} already downloaded. Skipping download.")
            
        # Read the raw XML directly and parse it
        with open(local_doc_path, "r", encoding="utf-8", errors="ignore") as xml_file:
            raw_xml = xml_file.read()
            
        result = parse_older_vote_xml(raw_xml, item_number)
        
        print(f"  -> Outcome: {result['outcome']} | Method: {result['method']}")
        if result['votes_favor'] is not None:
             print(f"  -> Votes: {result['votes_favor']} (+), {result['votes_against']} (-), {result['abstentions']} (O)")
        
        # Save to our final list
        extracted_older_data.append({
            "vote_id": vote_id,
            "master_doc": master_doc_id,
            "item_number": item_number,
            "method": result["method"],
            "outcome": result["outcome"],
            "votes_favor": result["votes_favor"],
            "votes_against": result["votes_against"],
            "abstentions": result["abstentions"]
        })
        
    except Exception as e:
        print(f"  -> Error processing {vote_id}: {e}")
        
    time.sleep(1)

df_older_votes = pd.DataFrame(extracted_older_data)
print("\n--- ALL OLDER VOTES EXTRACTED ---")
if not df_older_votes.empty:
    print(df_older_votes.head())

Scanning older JSON files...

Processing Item 64 from eli/dl/doc/PV-7-2014-04-03-VOT...
  -> PV-7-2014-04-03-VOT-FNL_en.xml already downloaded. Skipping download.
  -> Outcome: Unknown | Method: Unknown

Processing Item 1 from eli/dl/doc/PV-7-2014-04-03-VOT...
  -> PV-7-2014-04-03-VOT-FNL_en.xml already downloaded. Skipping download.
  -> Outcome: Approved | Method: Auto-adopted

Processing Item 1 from eli/dl/doc/PV-8-2016-04-14-VOT...
  -> PV-8-2016-04-14-VOT-FNL_en.xml already downloaded. Skipping download.
  -> Outcome: Approved | Method: Auto-adopted

Processing Item 2 from eli/dl/doc/PV-8-2016-04-14-VOT...
  -> PV-8-2016-04-14-VOT-FNL_en.xml already downloaded. Skipping download.
  -> Outcome: Approved | Method: Auto-adopted

Processing Item 7 from eli/dl/doc/PV-8-2016-04-28-VOT...
  -> PV-8-2016-04-28-VOT-FNL_en.xml already downloaded. Skipping download.
  -> Outcome: Approved | Method: Auto-adopted

Processing Item 8 from eli/dl/doc/PV-8-2016-04-28-VOT...
  -> PV-8-2016-04-28-VO

In [22]:
df_older_votes.head(20)


,vote_id,master_doc,item_number,method,outcome,votes_favor,votes_against,abstentions
0,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64,eli/dl/doc/PV-7-2014-04-03-VOT,64,Unknown,Unknown,NaN,NaN,NaN
1,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1,eli/dl/doc/PV-7-2014-04-03-VOT,1,Auto-adopted,Approved,NaN,NaN,NaN
2,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522221-1,eli/dl/doc/PV-8-2016-04-14-VOT,1,Auto-adopted,Approved,NaN,NaN,NaN
3,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522222-2,eli/dl/doc/PV-8-2016-04-14-VOT,2,Auto-adopted,Approved,NaN,NaN,NaN
4,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520210-7,eli/dl/doc/PV-8-2016-04-28-VOT,7,Auto-adopted,Approved,NaN,NaN,NaN
5,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520211-8,eli/dl/doc/PV-8-2016-04-28-VOT,8,Auto-adopted,Approved,NaN,NaN,NaN
6,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520212-9,eli/dl/doc/PV-8-2016-04-28-VOT,9,Roll Call / Electronic,Approved,88.0,534.0,11.0
7,eli/dl/event/MTG-PL-2016-06-09-VOT-ITM-540102-4,eli/dl/doc/PV-8-2016-06-09-VOT,4,Roll Call / Electronic,Approved,48.0,565.0,4.0


In [25]:
print(df_decisions.columns)
print(df_older_votes.columns)

Index(['vote_id', 'decision_id', 'start_date', 'method', 'outcome',
       'attendees', 'votes_favor', 'votes_against', 'heading',
       'absolute_majority'],
      dtype='str')
Index(['vote_id', 'master_doc', 'item_number', 'method', 'outcome',
       'votes_favor', 'votes_against', 'abstentions'],
      dtype='str')


In [27]:
df_vote_outcomes = pd.concat([df_decisions, df_older_votes], ignore_index=True)

In [28]:
len(df_vote_outcomes)

45

In [29]:
df_vote_outcomes.to_csv("vote_outcomes_2014-2026.csv")

In [8]:
#Getting more DEC items for votes of all weekdays and not just 2nd readings
df_all_votes = pd.read_csv("all_meetings_raw.csv")

In [9]:
# 1. Setup Folders
votes_json_folder = "all_days_votes_json_dumps"
decisions_json_folder = "all_daysdecisions_json_dumps"
os.makedirs(votes_json_folder, exist_ok=True)
os.makedirs(decisions_json_folder, exist_ok=True)

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

# 2. Extract and deduplicate all VOT links from the dataframe
unique_vot_ids = set()

for row_string in df_all_votes["consists_of"].dropna():
    try:
        uri_list = ast.literal_eval(row_string)
        for uri in uri_list:
            if "-VOT-" in uri:
                unique_vot_ids.add(uri)
    except (ValueError, SyntaxError):
        continue

print(f"Found {len(unique_vot_ids)} unique VOT items to fetch.")

# 3. The Unified API Loop
all_extracted_votes = []

for vote_id in unique_vot_ids:
    print(f"\nFetching VOT: {vote_id}...")
    
    # Initialize the complete blueprint for our data
    record = {
        "vote_id": vote_id,
        "decision_id": None,
        "start_date": None,  # <-- We will fill this from the DEC file now!
        "method": "Unknown",
        "outcome": "Unknown",
        "attendees": None,
        "votes_favor": None,
        "votes_against": None,
        "heading": None,
        "absolute_majority": False
    }
    
    url_vot = f"https://data.europarl.europa.eu/{vote_id}"
    
    try:
        # --- STEP A: Fetch the VOT item to find the DEC link ---
        resp_vot = requests.get(url_vot, headers=headers, timeout=15)
        
        if resp_vot.status_code == 200:
            vot_data = resp_vot.json()
            
            safe_vote_filename = vote_id.replace("/", "_") + ".json"
            with open(os.path.join(votes_json_folder, safe_vote_filename), "w", encoding="utf-8") as f:
                json.dump(vot_data, f, indent=4)
                
            if "data" in vot_data and len(vot_data["data"]) > 0:
                vot_item = vot_data["data"][0]
                
                # --- STEP B: Look inside for the DEC link and grab EVERYTHING ---
                if "consists_of" in vot_item and isinstance(vot_item["consists_of"], list):
                    for dec_id in vot_item["consists_of"]:
                        if "-DEC-" in dec_id:
                            print(f"  -> Found DEC: {dec_id}. Fetching results and timestamp...")
                            record["decision_id"] = dec_id
                            
                            resp_dec = requests.get(f"https://data.europarl.europa.eu/{dec_id}", headers=headers, timeout=15)
                            
                            if resp_dec.status_code == 200:
                                dec_data = resp_dec.json()
                                
                                safe_dec_filename = dec_id.split("/")[-1] + ".json"
                                with open(os.path.join(decisions_json_folder, safe_dec_filename), "w", encoding="utf-8") as f:
                                    json.dump(dec_data, f, indent=4)
                                    
                                if "data" in dec_data and len(dec_data["data"]) > 0:
                                    target_dict = dec_data["data"][0]
                                    heading_en = target_dict.get("headingLabel", {}).get("en", "")
                                    
                                    # Populate the rich data from the DEC file
                                    record["start_date"] = target_dict.get("activity_start_date") # <-- Timestamp extracted here!
                                    record["method"] = target_dict.get("decision_method", "Unknown")
                                    record["outcome"] = target_dict.get("decision_outcome", "Unknown")
                                    record["attendees"] = target_dict.get("number_of_attendees")
                                    record["votes_favor"] = target_dict.get("number_of_votes_favor")
                                    record["votes_against"] = target_dict.get("number_of_votes_against")
                                    record["heading"] = heading_en
                                    record["absolute_majority"] = "Rule 68" in heading_en or "Rule 69" in heading_en
                                    
                                    # Apply our "Approval without vote" patch
                                    if record["outcome"] == "Unknown" and "referenceText" in target_dict:
                                        ref_text = target_dict["referenceText"].get("en", "").lower()
                                        if "approval without vote" in ref_text or "declared approved" in ref_text:
                                            record["outcome"] = "Approved"
                                            record["method"] = "Auto-adopted"
                            
                            time.sleep(0.5)
                            break # Found the DEC, no need to keep checking the list
                            
        else:
            print(f"  -> Failed to fetch {vote_id}: Status {resp_vot.status_code}")
            
    except requests.exceptions.Timeout:
        print(f"  -> Warning: Request timed out for {vote_id}.")
    except Exception as e:
        print(f"  -> Error on {vote_id}: {e}")
        
    all_extracted_votes.append(record)
    time.sleep(0.5)

# 4. Build the final rich DataFrame
df_all_votes = pd.DataFrame(all_extracted_votes)
df_all_votes.to_csv("all_days_vote_results_w_time_stamps.csv", index=False)
print("\n--- EXTRACTION COMPLETE ---")
print(f"Successfully processed {len(df_all_votes)} votes.")
print(df_all_votes.head())

Found 1822 unique VOT items to fetch.

Fetching VOT: eli/dl/event/MTG-PL-2023-11-08-VOT-ITM-000001...
  -> Found DEC: eli/dl/event/MTG-PL-2023-11-08-DEC-160016. Fetching results and timestamp...

Fetching VOT: eli/dl/event/MTG-PL-2026-02-11-VOT-ITM-983762...
  -> Found DEC: eli/dl/event/MTG-PL-2026-02-11-DEC-184692. Fetching results and timestamp...

Fetching VOT: eli/dl/event/MTG-PL-2026-03-12-VOT-ITM-986076...
  -> Found DEC: eli/dl/event/MTG-PL-2026-03-12-DEC-186149. Fetching results and timestamp...

Fetching VOT: eli/dl/event/MTG-PL-2025-06-19-VOT-ITM-965710...
  -> Failed to fetch eli/dl/event/MTG-PL-2025-06-19-VOT-ITM-965710: Status 404

Fetching VOT: eli/dl/event/MTG-PL-2025-10-21-VOT-ITM-974693...
  -> Found DEC: eli/dl/event/MTG-PL-2025-10-21-DEC-179820. Fetching results and timestamp...

Fetching VOT: eli/dl/event/MTG-PL-2026-03-26-VOT-ITM-987012...
  -> Found DEC: eli/dl/event/MTG-PL-2026-03-26-DEC-189604. Fetching results and timestamp...

Fetching VOT: eli/dl/event/MTG-PL